# Introduction to Human Language Technologies
## Lab 5: Lexical Semantics



In [2]:
from functools import lru_cache
from itertools import combinations

import nltk
import pandas as pd
from nltk.corpus import wordnet as wn, wordnet_ic

for resource in ('wordnet', 'wordnet_ic'):
    try:
        nltk.data.find(f'corpora/{resource}')
    except LookupError:
        try:
            nltk.data.find(f'corpora/{resource}.zip')
        except LookupError:
            if not nltk.download(resource, quiet=True):
                raise RuntimeError(f'Could not download {resource}')

brown_ic = wordnet_ic.ic('ic-brown.dat')

### Data Preparation

In [3]:
sentences = [
    [('the', 'DT'), ('man', 'NN'), ('swim', 'VB'), ('with', 'PR'),
     ('a', 'DT'), ('girl', 'NN'), ('and', 'CC'), ('a', 'DT'),
     ('boy', 'NN'), ('whilst', 'PR'), ('the', 'DT'), ('woman', 'NN'),
     ('walk', 'VB')],
    [('the', 'DT'), ('dog', 'NN'), ('run', 'VB'), ('after', 'PR'),
     ('a', 'DT'), ('cat', 'NN')]
]

pos_map = {'N': wn.NOUN, 'V': wn.VERB, 'J': wn.ADJ, 'R': wn.ADV}

@lru_cache(maxsize=None)
def most_frequent_synset(lemma, tag):
    pos = pos_map.get(tag[0])
    if pos is None:
        return None
    synsets = wn.synsets(lemma, pos=pos)
    return synsets[0] if synsets else None

### Most frequent synsets

In [4]:
for number, sentence in enumerate(sentences, start=1):
    print('\nSentence', number)
    for lemma, tag in sentence:
        synset = most_frequent_synset(lemma, tag)
        name = synset.name() if synset is not None else 'N/A'
        print(lemma, tag, name)

content = {}
for sentence in sentences:
    for lemma, tag in sentence:
        synset = most_frequent_synset(lemma, tag)
        if synset is not None:
            content[(lemma, tag)] = synset


Sentence 1
the DT N/A
man NN man.n.01
swim VB swim.v.01
with PR N/A
a DT N/A
girl NN girl.n.01
and CC N/A
a DT N/A
boy NN male_child.n.01
whilst PR N/A
the DT N/A
woman NN woman.n.01
walk VB walk.v.01

Sentence 2
the DT N/A
dog NN dog.n.01
run VB run.v.01
after PR N/A
a DT N/A
cat NN cat.n.01


### LCS and Similarities



In [5]:
lch_max = {}
for synset in content.values():
    if synset.pos() not in lch_max:
        lch_max[synset.pos()] = synset.lch_similarity(synset)

def compare_synsets(first, second):
    if first.pos() != second.pos() or first.pos() not in ('n', 'v'):
        return None
    lcs = sorted(s.name() for s in first.lowest_common_hypernyms(second))
    lch = first.lch_similarity(second)
    return {
        'lcs': lcs,
        'path': first.path_similarity(second),
        'lch': lch,
        'lch_norm': lch / lch_max[first.pos()] if lch is not None else None,
        'wup': first.wup_similarity(second),
        'lin': first.lin_similarity(second, brown_ic)
    }

results = []
skipped = 0
for (word1, synset1), (word2, synset2) in combinations(content.items(), 2):
    scores = compare_synsets(synset1, synset2)
    if scores is None:
        skipped += 1
        continue
    results.append({'words': (word1[0], word2[0]), **scores})

In [6]:
print(len(content), 'content words')
print(len(results), 'comparable pairs')
print(skipped, 'noun/verb pairs skipped')

table = pd.DataFrame([{
    'Word 1': result['words'][0],
    'Word 2': result['words'][1],
    'LCS': ', '.join(result['lcs']),
    'Path': result['path'],
    'LCH': result['lch'],
    'LCH norm': result['lch_norm'],
    'WUP': result['wup'],
    'Lin': result['lin']
} for result in results])

table.round(3)

9 content words
18 comparable pairs
18 noun/verb pairs skipped


,Word 1,Word 2,LCS,Path,LCH,LCH norm,WUP,Lin
0,man,girl,adult.n.01,0.250,2.251,0.619,0.632,0.714
1,man,boy,male.n.02,0.333,2.539,0.698,0.667,0.729
2,man,woman,adult.n.01,0.333,2.539,0.698,0.667,0.787
3,man,dog,organism.n.01,0.143,1.692,0.465,0.667,0.292
4,man,cat,organism.n.01,0.083,1.153,0.317,0.522,0.291
5,swim,walk,travel.v.01,0.333,2.159,0.663,0.333,0.491
6,swim,run,travel.v.01,0.250,1.872,0.575,0.286,0.465
7,girl,boy,person.n.01,0.167,1.846,0.507,0.632,0.293
8,girl,woman,woman.n.01,0.500,2.944,0.809,0.632,0.907
9,girl,dog,organism.n.01,0.125,1.558,0.428,0.632,0.269


*What similarity seems better?*

Lin similarity seems to be the best one. It gives the highest scores to pairs that are really close in meaning (girl–woman 0.91, dog–cat 0.88, man–woman 0.79, man–boy 0.73) and low scores (around 0.25–0.29) to every human–animal pair, so the gap between related and unrelated words is clear. This is because it uses information content from the Brown corpus: a very general LCS like `organism.n.01` gives a low score, while a specific one like `carnivore.n.01` or `woman.n.01` gives a high one.

The other measures only look at the WordNet graph. Path and LCH depend only on the path length, so they rank the pairs in the same way, and both give dog–cat (0.20 and 0.56) a lower score than man–woman (0.33 and 0.70). Wu-Palmer gives man–dog the same score as man–woman (0.667), so it barely separates people from animals.

Lin is not perfect: girl–boy (0.29) is almost as low as girl–dog (0.27), because their LCS is `person.n.01`, which is quite general. Even so, it is the measure that best matches our intuition overall.